# Problem 1 — Continuation: Model Selection & Final Evaluation

> **Continue from `problem1_phase2_completed.ipynb`.** Run that notebook first, then run every cell below in order. This notebook assumes the trained models and arrays (`X_val_flat`, `y_val`, `X_test_flat`, `y_test`) are already available.

This section completes the workflow after the Phase 2 validation comparison. The validation set is used for model selection; the test set is reserved for one final evaluation.

## Step 1 — Select the best model using validation metrics

We compare models using more than accuracy. The code below selects the model with the highest **validation MCC** as a default selection criterion (MCC considers all classes and is useful when class distribution is uneven). Review the complete table and justify the choice in your report; you may choose another metric if the assignment specifies one.

In [ ]:
# Re-create the model registry (these models should already be trained)
models = {
    "Logistic Regression": lr_model,
    "Decision Tree": dt_model,
    "KNN": knn_model,
    "Naive Bayes": nb_model,
    "SVM": svm_model,
}

# Sort validation results by MCC (then F1 as a tie-breaker)
validation_ranked = results_df.sort_values(
    by=["MCC", "F1"], ascending=False
).reset_index(drop=True)

display(validation_ranked.round(4))

best_model_name = validation_ranked.loc[0, "Model"]
best_model = models[best_model_name]

print("Selected model:", best_model_name)
print("Selection criterion: highest validation MCC (F1 tie-breaker)")

## Step 2 — Confusion matrix for the selected model

A confusion matrix shows which activities are correctly classified and which activities are confused with one another. Rows are true labels and columns are predicted labels.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

activity_names = [
    "Walking", "Walking Upstairs", "Walking Downstairs",
    "Sitting", "Standing", "Laying"
]
class_labels = np.arange(1, 7)

y_val_pred_best = best_model.predict(X_val_flat)
cm_val = confusion_matrix(y_val, y_val_pred_best, labels=class_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=activity_names
)
fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"Validation Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

print("Validation confusion matrix:")
display(pd.DataFrame(cm_val, index=activity_names, columns=activity_names))

## Step 3 — Final evaluation on the test set (once)

**Do not use test metrics to change/select the model.** After validation-based selection, evaluate the chosen model on the held-out test set exactly once.

In [ ]:
test_result = evaluate_model(
    best_model, X_test_flat, y_test,
    f"{best_model_name} (Final Test)"
)

test_results_df = pd.DataFrame([test_result])
display(test_results_df.round(4))

y_test_pred_best = best_model.predict(X_test_flat)
cm_test = confusion_matrix(y_test, y_test_pred_best, labels=class_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_test,
    display_labels=activity_names
)
fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"Test Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## Step 4 — Save the selected model and evaluation outputs

The files are written into `problem1/artifacts/`. Keep these artifacts with your submission/repository as required.

In [ ]:
from pathlib import Path
import joblib

ARTIFACTS_DIR = Path("problem1/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACTS_DIR / "best_har_model.joblib"
joblib.dump(best_model, model_path)

validation_ranked.to_csv(ARTIFACTS_DIR / "validation_model_comparison.csv", index=False)
test_results_df.to_csv(ARTIFACTS_DIR / "final_test_metrics.csv", index=False)
pd.DataFrame(cm_test, index=activity_names, columns=activity_names).to_csv(
    ARTIFACTS_DIR / "test_confusion_matrix.csv"
)

print("Saved model:", model_path)
print("Saved validation comparison, final test metrics, and test confusion matrix.")

## Step 5 — Write your interpretation (complete before submission)

Use the actual values printed above; do not invent or copy example numbers.

- **Model selected:** [name]
- **Why selected (validation):** [MCC, F1, AUC, and metric trade-offs]
- **Class imbalance:** [describe class counts from the earlier dataset section]
- **Validation confusion matrix:** [which activities are confused?]
- **Final test performance:** [accuracy, macro AUC, weighted precision/recall/F1, MCC]
- **Generalization:** [compare validation and test results; discuss possible overfitting]
- **Limitations:** [flattening loses temporal structure; discuss any other limitations]

### Optional: inspect class distribution

This is descriptive only and does not alter model selection.

In [ ]:
print("Training label counts:")
display(pd.Series(y_train).value_counts().sort_index().rename(index=dict(zip(range(1, 7), activity_names))).to_frame("count"))

print("Validation label counts:")
display(pd.Series(y_val).value_counts().sort_index().rename(index=dict(zip(range(1, 7), activity_names))).to_frame("count"))

print("Test label counts:")
display(pd.Series(y_test).value_counts().sort_index().rename(index=dict(zip(range(1, 7), activity_names))).to_frame("count"))

## Step 6 — Submission/repository checklist

- [ ] Notebook runs top-to-bottom without errors.
- [ ] Validation comparison includes all five classifiers and required metrics.
- [ ] Model choice is based on validation data and explained.
- [ ] Confusion matrix is included and interpreted.
- [ ] Test set was used only after model selection.
- [ ] Selected model and metric outputs are saved.
- [ ] `requirements.txt` lists required packages (e.g., `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `torch`, `joblib`).
- [ ] README documents dataset, preprocessing, models, metrics, results, and how to run.
- [ ] GitHub repository contains notebook, README, requirements, and relevant figures/artifacts.
- [ ] If required, Streamlit app loads the saved model and demonstrates prediction.

## Important
This continuation expects the exact variables created in the preceding notebook. If you renamed any variables, adjust the corresponding references here. The model is selected using validation MCC by default; follow your instructor's rubric if it specifies a different selection rule.